# Lab 3 Demo — Track B: Offer Generation (RAG)

Pipeline (`rag/offer_rag.py`): header-based markdown chunks of `northstar-policy-docs/` → Titan Text Embeddings v2 (1024-d, normalised) → FAISS `IndexFlatIP` (cosine) → top-6 chunks → Claude Haiku 4.5 (`us.` inference profile, temperature 0.2) → 2–3 sentence retention offer.

Run with **Kernel → Restart & Run All** from inside `notebooks/`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../rag'))
import ragas, faiss, boto3
print('ragas', ragas.__version__, '| faiss', faiss.__version__, '| boto3', boto3.__version__)

c:\Users\jcdah\anaconda3\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


ragas 0.4.3 | faiss 1.15.1 | boto3 1.43.111


## 1. Build the index (once)

In [2]:
from offer_rag import get_index, answer_customer_rag, LLM_MODEL_ID, MODEL_ID
docs, index = get_index()
print(f'Embedding model: {MODEL_ID}\nGenerator: {LLM_MODEL_ID}\nChunks indexed: {index.ntotal}')
for d in docs[:5]:
    print(' ', d.metadata['id'], '|', d.page_content.splitlines()[0][:60])

File: customer-faq.md (6208 characters)
File: loyalty-program-terms.md (5960 characters)
File: return-policy.md (6185 characters)
File: shipping-policy.md (4569 characters)
Total vectors indexed: 48
Embedding model: amazon.titan-embed-text-v2:0
Generator: us.anthropic.claude-haiku-4-5-20251001-v1:0
Chunks indexed: 48
  customer-faq#0 | # NorthStar Retail — Customer Service FAQ  
  customer-faq#1 | ## Returns  
  customer-faq#2 | ## Loyalty  
  customer-faq#3 | ## Shipping  
  customer-faq#4 | ## Orders and Account  


## 2. Generate offers: the 4 harness customers plus one extra

In [3]:
import evaluation_harness as eh
extra = {
    'churn_probability': 0.69, 'loyalty_tier': 'Platinum', 'days_since_last_purchase': 33,
    'total_lifetime_value': 7400.0, 'avg_order_value': 410.0, 'category_diversity_score': 0.6,
    'online_to_store_ratio': 0.4, 'top_categories': ['Camping', 'Outerwear'],
}
cases = [(tc.customer_id, tc.customer_context, tc.question) for tc in eh.OFFER_TEST_CASES]
cases.append(('CUST-DEMO-EXTRA', extra,
              'Generate a retention offer for this at-risk Platinum member who buys camping gear and outerwear.'))

demo_outputs = []
for cid, ctx, q in cases:
    r = answer_customer_rag(ctx, q)
    demo_outputs.append((cid, ctx, r))
    print(f"=== {cid} | tier={ctx['loyalty_tier']} | churn_p={ctx['churn_probability']} | top={ctx['top_categories']}")
    print(r['answer'])
    print('retrieved chunks:', r['chunk_ids'], '\n')

[Usage] Input tokens: 1359, Output tokens: 99
=== CUST-10000317 | tier=Gold | churn_p=0.81 | top=['Outdoor', 'Footwear']
As a Gold member, you're earning 1.5× points on every purchase and enjoying free standard shipping on all orders—plus your recent activity qualifies you for early 24-hour access to our upcoming Outdoor gear sale where you can maximize those points faster. We'd love to see you back in action: redeem your accumulated points toward premium Footwear this week and lock in an extra 500 bonus points to help you stay engaged with us.
retrieved chunks: ['loyalty-program-terms#9', 'customer-faq#2', 'loyalty-program-terms#2', 'loyalty-program-terms#5', 'loyalty-program-terms#6', 'loyalty-program-terms#3'] 

[Usage] Input tokens: 1486, Output tokens: 85
=== CUST-10000904 | tier=Bronze | churn_p=0.74 | top=['Apparel']
As a Bronze member, you're earning 1 point per $1 on apparel purchases, and we'd love to welcome you back with a $10 birthday reward if your birthday is coming up s

## 3. RAGAS evaluation (one run here; the graded multi-run results are produced by `rag/run_eval.py` → `docs/lab3-evidence/rag_eval.csv`)

In [4]:
df = eh.RAGEvaluator(rag_invoke_fn=answer_customer_rag).run(eh.OFFER_TEST_CASES)
df.insert(0, 'case', [tc.customer_id for tc in eh.OFFER_TEST_CASES])
print('Judge model:', eh.DEFAULT_BEDROCK_MODEL_ID)
df[['case', 'faithfulness', 'answer_relevancy', 'context_recall']].round(3)


── Running RAG Evaluation (4 test cases) ──
  Evaluating CUST-10000317...
[Usage] Input tokens: 1359, Output tokens: 95
  Evaluating CUST-10000904...
[Usage] Input tokens: 1486, Output tokens: 73
  Evaluating CUST-10000122...
[Usage] Input tokens: 1402, Output tokens: 103
  Evaluating CUST-10000588...
[Usage] Input tokens: 1438, Output tokens: 90


Evaluating:   0%|          | 0/12 [00:00<?, ?it/s]


── RAGAS Evaluation Results ──────────────────────────
  faithfulness        : 0.469  (target: 0.80)  ✗ FAIL
  answer_relevancy    : 0.351  (target: 0.75)  ✗ FAIL
  context_recall      : 0.608  (target: 0.70)  ✗ FAIL

Judge model: us.anthropic.claude-haiku-4-5-20251001-v1:0


,case,faithfulness,answer_relevancy,context_recall
0,CUST-10000317,0.250,0.312,1.000
1,CUST-10000904,0.400,0.383,0.600
2,CUST-10000122,0.600,0.357,0.333
3,CUST-10000588,0.625,0.352,0.500


In [5]:
import pandas as pd
allr = pd.read_csv('../evaluation_results/rag_eval_all.csv')
m = ['faithfulness', 'answer_relevancy', 'context_recall']
allr.groupby('config')[m].agg(['mean', 'min', 'max']).round(3)

faithfulness               answer_relevancy                \
                 mean    min    max             mean    min    max   
config                                                               
baseline        0.412  0.000  0.625            0.376  0.343  0.464   
improved        0.521  0.375  0.875            0.369  0.328  0.421   

         context_recall               
                   mean    min   max  
config                                
baseline          0.496  0.333  0.75  
improved          0.608  0.333  1.00